In [1]:
!git clone https://github.com/Morteza-24/llm-uncertainty-head.git
!mv llm-uncertainty-head/* llm-uncertainty-head/.[!.]* ./
%pip install git+https://github.com/IINemo/lm-polygraph.git
%pip install -e .

Cloning into 'llm-uncertainty-head'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 304 (delta 89), reused 98 (delta 63), pack-reused 155 (from 1)
Receiving objects: 100% (304/304), 691.30 KiB | 17.28 MiB/s, done.
Resolving deltas: 100% (168/168), done.
  Cloning https://github.com/IINemo/lm-polygraph.git to /tmp/pip-req-build-4338ai8z
  Running command git clone --filter=blob:none --quiet https://github.com/IINemo/lm-polygraph.git /tmp/pip-req-build-4338ai8z
  Resolved https://github.com/IINemo/lm-polygraph.git to commit efea882d810d07770e71d3a80e02416d09751435
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 11.5 MB/

# Train

In [1]:
!CUDA_VISIBLE_DEVICES=0 python -m luh.cli.train.run_train_uhead --config-dir=./configs --config-name=run_train_uhead.yaml dataset.path="hf:llm-uncertainty-head/train_akimbio_mistral" model.pretrained_model_name_or_path="mistralai/Mistral-7B-Instruct-v0.2"

[2026-08-19 03:49:53,753][root][INFO] - Output directory: /content/workdir/train/2026-08-19/03-49-53
[2026-08-19 03:49:53,753][transformers][INFO] - Init transformers logger.
[2026-08-19 03:49:53,756][root][INFO] - Loading model...
[2026-08-19 03:49:53,756][root][INFO] - Loading model mistralai/Mistral-7B-Instruct-v0.2...
[2026-08-19 03:49:53,932][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-08-19 03:49:53,932][huggingface_hub.utils._http][WARNING] - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-08-19 03:49:53,941][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
[2026-08-19 03:49:53,942][transformers.configuration_utils][INFO] - loading confi

# Infer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from luh import AutoUncertaintyHead
from lm_polygraph import CausalLMWithUncertainty
from luh.calculator_infer_luh import CalculatorInferLuh
from luh.calculator_apply_uq_head import CalculatorApplyUQHead
from luh.luh_estimator_dummy import LuhEstimatorDummy

In [ ]:
# load model and uhead
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
uhead_name = "llm-uncertainty-head/uhead6_Mistral-7B-Instruct-v0.2"
# uhead_name = "/content/workdir/train/2026-08-19/03-49-53/model"

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(
    model_name)
tokenizer.pad_token = tokenizer.eos_token
uhead = AutoUncertaintyHead.from_pretrained(
    uhead_name, base_model=llm)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

config.yaml:   0%|          | 0.00/241 [00:00<?, ?B/s]

weights.pth: reconstructing file:   0%|          |  0.00B / 19.7MB            

weights.pth: downloading bytes:           |  0.00B            

In [ ]:
generation_config = GenerationConfig.from_pretrained(model_name)
args_generate = {"generation_config": generation_config,
                 "max_new_tokens": 50}
calc_infer_llm = CalculatorInferLuh(uhead,
                                    tokenize=True,
                                    args_generate=args_generate,
                                    device="cuda",
                                    generations_cache_dir="",
                                    predict_token_uncertainties=True)

estimator = LuhEstimatorDummy()
llm_adapter = CausalLMWithUncertainty(llm, tokenizer=tokenizer, stat_calculators=[calc_infer_llm], estimator=estimator)

In [ ]:
# prepare text ...

messages = [
    [
        {
            "role": "user",
            "content": "In which year did the programming language Mercury first appear? Answer with a year only."
        }
    ]
]
# The correct answer is 1995

chat_messages = [tokenizer.apply_chat_template(m, tokenize=False, add_bos_token=False) for m in messages]
inputs = tokenizer(chat_messages, return_tensors="pt", padding=True, truncation=True, add_special_tokens=False).to("cuda")

output = llm_adapter.generate(inputs["input_ids"])
output["uncertainty_score"]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=46) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


[[0.5005925893783569,
  0.11431393772363663,
  0.19080078601837158,
  0.6520728468894958,
  0.430184930562973,
  0.12376189231872559,
  0.3136047422885895,
  0.5520869493484497,
  0.4270686209201813,
  0.36228707432746887,
  0.5659871101379395,
  0.814650297164917,
  0.6513360738754272,
  0.7492914199829102,
  0.7665389776229858,
  0.8988367319107056,
  0.7635713219642639,
  0.7144246697425842,
  0.5759887099266052,
  0.7082247734069824]]

In [ ]:
print("Model response and uncertainty scores:")
print(f'Response: {tokenizer.batch_decode(output["sequences"][:,len(inputs["input_ids"][0]):])}')
print(f'UE Scores: {output["uncertainty_score"][0]}')


Model response and uncertainty scores:
Response: ['Mercury is a logic programming language that was first announced in 1993. However,']
UE Scores: [0.5005925893783569, 0.11431393772363663, 0.19080078601837158, 0.6520728468894958, 0.430184930562973, 0.12376189231872559, 0.3136047422885895, 0.5520869493484497, 0.4270686209201813, 0.36228707432746887, 0.5659871101379395, 0.814650297164917, 0.6513360738754272, 0.7492914199829102, 0.7665389776229858, 0.8988367319107056, 0.7635713219642639, 0.7144246697425842, 0.5759887099266052, 0.7082247734069824]


In [ ]:
def highlight_html_tokens(
    token_ids,
    positions_to_highlight,
    tokenizer,
    color="red",
    font_weight="bold"
):
    """
    Convert a list of token IDs into a readable string, highlight tokens at
    the specified positions in `positions_to_highlight`, and remove the leading
    '▁' that Mistral/Llama tokenizers use for word boundaries.

    Args:
        token_ids (List[int]): The sequence of token IDs.
        tokenizer: A Hugging Face tokenizer (e.g., for mistralai/Mistral-7B-Instruct-v0.2).
        positions_to_highlight (Set[int] or List[int]): 0-based indices of tokens to highlight.
        color (str): CSS color for the highlighted text (default "red").
        font_weight (str): CSS font weight (default "bold").

    Returns:
        str: An HTML string with some tokens highlighted.
    """
    # Convert the IDs to subword tokens (may contain leading "▁")
    raw_tokens = tokenizer.convert_ids_to_tokens(token_ids)

    # Ensure positions_to_highlight is a set for quick membership check
    if not isinstance(positions_to_highlight, set):
        positions_to_highlight = set(positions_to_highlight)

    final_pieces = []

    for idx, token in enumerate(raw_tokens):
        # If the token starts with "▁", replace that with a literal space
        if token.startswith("▁"):
            display_str = " " + token[1:]
        else:
            display_str = token

        # If this position is in positions_to_highlight, wrap in <span>
        if idx in positions_to_highlight:
            display_str = (
                f"<span style='color:{color}; font-weight:{font_weight};'>"
                f"{display_str}"
                "</span>"
            )

        final_pieces.append(display_str)

    # Join everything without extra spaces
    return "".join(final_pieces)

In [ ]:
from IPython.display import HTML


def highlight_uncertain_claims(uncertainties, generated_tokens, claims):
    threshold = 0.5
    tokens_to_highlight = set()

    for ue_score, claim in zip(uncertainties, claims):
        if ue_score > threshold:
            tokens_to_highlight.update([claim])

    display(HTML(highlight_html_tokens(generated_tokens, tokens_to_highlight, tokenizer)))

In [ ]:
highlight_uncertain_claims(
    output["uncertainty_score"][0],
    output["sequences"][:,len(inputs["input_ids"][0]):][0],
    list(range(len(output["sequences"][:,len(inputs["input_ids"][0]):][0]))),
)